# Goals:

## Data Goal
- Import penguins from sns library
- Set aside the 9 records with no Nan for sex
- Remove the other 2 records with NaN
- Files:
    - penguins_clean.csv (333 records)
    - penguins_full.csv (342 records)
    - penguins_unknown_sex.csv (9 records)

## Data Prep Goal
- Identify the following:
    - Model
    - Scaler

## 1st Classification:
-Create a model that will be used to determine the best sex for each of the 9 penguins with unknown sex

## Classification Pipeline
- Do the following:
    - Using the best Model and scaler technique to create:
        - a pipeline for classification
        - target being sex
        - use the penguins_full.csv (342 records) 

In [43]:
import pandas as pd
import numpy as np
import seaborn as sns

In [44]:
#sns.get_dataset_names()
df = sns.load_dataset('penguins')
print(df.shape)

(344, 7)


In [45]:
df.isna().sum()

species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64

In [46]:
penguins_unknown_sex = df[
    df["sex"].isna() &
    df["bill_length_mm"].notna() &
    df["bill_depth_mm"].notna() &
    df["flipper_length_mm"].notna() &
    df["body_mass_g"].notna()
].copy()

penguins_unknown_sex.shape

(9, 7)

In [47]:
penguins_unknown_sex.to_csv("penguins_unknown_sex.csv", index=False)

In [48]:
penguins_full = df.dropna(
    subset=[
        "bill_length_mm",
        "bill_depth_mm",
        "flipper_length_mm",
        "body_mass_g"
    ]
).copy()

penguins_full.shape

(342, 7)

In [49]:
penguins_full.to_csv("penguins_full.csv", index=False)

In [50]:
penguins_clean = df.dropna().copy()
penguins_clean.shape

(333, 7)

In [51]:
penguins_clean.to_csv("penguins_clean.csv", index=False)

In [52]:
print("Clean:", penguins_clean.shape)
print("Full:", penguins_full.shape)
print("Unknown sex:", penguins_unknown_sex.shape)

333 + 9

Clean: (333, 7)
Full: (342, 7)
Unknown sex: (9, 7)


342

In [53]:
pd.read_csv('./penguins_clean.csv').value_counts()

species  island     bill_length_mm  bill_depth_mm  flipper_length_mm  body_mass_g  sex   
Adelie   Biscoe     34.5            18.1           187.0              2900.0       Female    1
Gentoo   Biscoe     44.0            13.6           208.0              4350.0       Female    1
                    43.6            13.9           217.0              4900.0       Female    1
                    43.5            15.2           213.0              4650.0       Female    1
                                    14.2           220.0              4700.0       Female    1
                                                                                            ..
Adelie   Torgersen  36.6            17.8           185.0              3700.0       Female    1
                    36.2            17.2           187.0              3150.0       Female    1
                                    16.1           187.0              3550.0       Female    1
                    35.9            16.6           190.

In [54]:
X.isna().sum()

bill_length_mm       2
bill_depth_mm        2
flipper_length_mm    2
body_mass_g          2
species_enc          0
dtype: int64

In [35]:

from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score


In [36]:

penguins_clean = pd.read_csv("./data/penguins_clean.csv")
df = penguins_clean.copy()

In [37]:
le_species = LabelEncoder()
le_sex = LabelEncoder()

df["species_enc"] = le_species.fit_transform(df["species"])
df["sex_enc"] = le_sex.fit_transform(df["sex"])


In [38]:
feature_cols = [
    "bill_length_mm",
    "bill_depth_mm",
    "flipper_length_mm",
    "body_mass_g",
    "species_enc"
]

X = df[feature_cols]
y = df["sex_enc"]

print("Feature matrix shape:", X.shape)
print("Target distribution:\n", y.value_counts())

Feature matrix shape: (344, 5)
Target distribution:
 sex_enc
1    168
0    165
2     11
Name: count, dtype: int64


In [39]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [40]:
pipelines = {
    "LogReg (no scaling)": Pipeline([
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "LogReg + StandardScaler": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "LogReg + MinMaxScaler": Pipeline([
        ("scaler", MinMaxScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "LogReg + RobustScaler": Pipeline([
        ("scaler", RobustScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]),
    
    "Random Forest": Pipeline([
        ("model", RandomForestClassifier(random_state=42))
    ])
}

In [41]:
results = []

for name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    test_score = pipe.score(X_test, y_test)
    
    results.append({
        "model": name,
        "test_accuracy": test_score
    })

results_df = pd.DataFrame(results).sort_values(
    by="test_accuracy", ascending=False
)

results_df


ValueError: Input X contains NaN.
LogisticRegression does not accept missing values encoded as NaN natively. For supervised learning, you might want to consider sklearn.ensemble.HistGradientBoostingClassifier and Regressor which accept missing values encoded as NaNs natively. Alternatively, it is possible to preprocess the data, for instance by using an imputer transformer in a pipeline or drop samples with missing values. See https://scikit-learn.org/stable/modules/impute.html You can find a list of all estimators that handle NaN values at the following page: https://scikit-learn.org/stable/modules/impute.html#estimators-that-handle-nan-values